# Notebook for PSF denoising with a PSF generator
## Application to 4 different observations

In [ ]:
import torch
import os
from os import listdir
from os.path import isfile, join
import sys
from time import time
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import scipy
import scipy.optimize

grad = torch.autograd.grad

In [ ]:
sys.path.insert(1, "../training")
from wgan_gradient_penalty import *

## Log-equalizations

In [ ]:
# Log
def logeq2(Im):
    return np.log(10000*Im+1)/np.log(10001)+0.3
def logeq2_torch(Im):
    return torch.log(10000*Im+1)/np.log(10001)+0.3
def delogeq2(Im):
    return (10001**(Im-0.3)-1)/10000
def logeq(Im):
    return np.log(1000*Im+1)/np.log(1001)

# Log for visualization
def normminmax(Im):
    return (Im - np.min(Im))/(np.max(Im) - np.min(Im))
def viseq(Im):
    return logeq(normminmax(Im))



## Getting the PSF to denoise

In [ ]:
# Name of the PSF to denoise

#to_test = "pds70"
#to_test = "sao206462"
to_test = "hip72192"
#to_test = "hip80019"

path_datas = "../../data/observations_PSF/"
if "hip" in to_test:
    path_PSFs = path_datas + to_test + "_ird_sortframes_dc-IRD_SCIENCE_PSF_MASTER_CUBE-median_unsat.fits"
else:
    path_PSFs = path_datas + to_test + "_ird_convert_recenter_dc5-IRD_SCIENCE_PSF_MASTER_CUBE-median_unsat.fits"


In [ ]:
fits_PSFs = fits.open(path_PSFs)
images_PSFs = fits_PSFs[0].data

# First dimension : filter
# Second dimension : frame

In [ ]:
fits_PSFs[0].header

In [ ]:
fits_PSFs.info()

In [ ]:
# Number of frames available
if len(images_PSFs.shape) == 4:
    nb_frames= 2
else:
    nb_frames = 1

In [ ]:
# Showing the PSF
if nb_frames > 1:
    plt.imshow(viseq(images_PSFs[0,0]))
    plt.show()
    plt.imshow(viseq(images_PSFs[0,1]))
    plt.show()
    plt.imshow(viseq(images_PSFs[1,0]))
    plt.show()
    plt.imshow(viseq(images_PSFs[1,1]))
    plt.show()
else:
    plt.imshow(viseq(images_PSFs[0]))
    plt.show()
    plt.imshow(viseq(images_PSFs[1]))
    plt.show()

In [ ]:
PSF_og = {} # PSF to denoise
PSF_gen = {} # PSF denoised
z_gen = {} # Corresponding latent vector
best_MSEs = {} # Corresponding distances between original and denoised PSF

In [ ]:
# Store the frames in the dictionary

if nb_frames > 1:
    PSF_og["K1_f1"] = images_PSFs[0,0]
    PSF_og["K1_f2"] = images_PSFs[0,1]
    PSF_og["K2_f1"] = images_PSFs[1,0]
    PSF_og["K2_f2"] = images_PSFs[1,1]
    L_frames = ["K1_f1", "K1_f2", "K2_f1", "K2_f2"]
else:
    
    PSF_og["K1"] = images_PSFs[0]
    PSF_og["K2"] = images_PSFs[1]
    
    L_frames = ["K1", "K2"]

# First step : recentering the 0

In [ ]:
criterias = {} # Percentile used to estimate the 0
alphas = {} # Sum of psf. We need it stored for unequalizing later.
shifts = {}
maxs = {} # Max of the psf. We need it stored for unequalizing later.

for name_frame in L_frames:
    criterias[name_frame] = 35

In [ ]:
#if nb_frames > 1:
#    criterias["K1_f1"] = 30
#    criterias["K1_f2"] = 30
#    criterias["K2_f1"] = 40
#    criterias["K2_f2"] = 45
#else:
#    criterias["K1"] = 40
#    criterias["K2"] = 40

In [ ]:
# Visual check that the 0 is well estimated
for frame in L_frames:
    print(frame, )
    plt.hist(PSF_og[frame].flatten()[PSF_og[frame].flatten()<200], bins=100)
    plt.plot([np.percentile(PSF_og[frame].flatten(), criterias[frame]), np.percentile(PSF_og[frame].flatten(), criterias[frame])], [0, 500])
    plt.show()

In [ ]:
PSF_old = PSF_og.copy()

In [ ]:
for frame in L_frames:
    shifts[frame] = np.percentile(PSF_og[frame].flatten(), criterias[frame]) # Store the shift
    PSF_og[frame] = PSF_og[frame] - np.percentile(PSF_og[frame].flatten(), criterias[frame])
    print("Sum of the PSF : ",np.sum(PSF_og[frame].flatten()), "\n")
    alphas[frame] = np.sum(PSF_og[frame].flatten())
    maxs[frame] = np.max(PSF_og[frame].flatten())
    PSF_og[frame] = PSF_og[frame] / np.sum(PSF_og[frame].flatten()) # Equalizing

In [ ]:
# Show the psf in different shapes (log-equalized or not)
for frame in L_frames:
    
    print(frame)
    print("Original domain")
    plt.imshow(PSF_og[frame])
    plt.colorbar()
    plt.show()
    print("--------------------")
    
    print("Log visualization")
    plt.imshow(viseq(PSF_og[frame]))
    plt.colorbar()
    plt.show()
    print("--------------------")
    
    print("Log equalization")
    plt.imshow(logeq2(PSF_og[frame]))
    plt.colorbar()
    plt.show()
    print("--------------------\n-------------------")

## Generator initialization

In [ ]:
ngpu=1
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
nc = 1
ndf = 128//4
n_dim=128

nz=20
training_s = "1719"
mode_eq = "log"
year = "2025"

white_gen = (mode_eq == "white")

In [ ]:
def MSE(A,B):
    return np.sum(np.nan_to_num(A-B)**2)/np.sum((1-np.isnan(A-B)))
def crop(A, crop_size=64, og_size=128): # To crop to the fits size
    return A[og_size//2-crop_size//2:og_size//2+crop_size//2, og_size//2-crop_size//2:og_size//2+crop_size//2]
def crop_tensor4(A, crop_size=64, og_size=128): # Same thing with tensor in a batch
    return A[:,:,og_size//2-crop_size//2:og_size//2+crop_size//2, og_size//2-crop_size//2:og_size//2+crop_size//2]

In [ ]:
racine = "../../"
path_models_W = racine + "models/"

name_model_W = "generatorK12_"+year+"_train"+training_s+"_nlat"+str(nz)+"_"+mode_eq+".pkl"


path_MEAN_STD = "../data/statistics/"

In [ ]:
if white_gen:
    
    if training_s =="1719":
        suffix = "_k12"
    else:
        suffix = "_h23"
        
    MEAN = np.load(path_MEAN_STD+"MEAN_"+training_s+suffix+".npy")
    STD = np.load(path_MEAN_STD+"STD_"+training_s+suffix+".npy")
    
    MEAN_cuda= torch.Tensor(MEAN).to(device)
    STD_cuda= torch.Tensor(STD).to(device)
    
    bound_mini = -3.5
    bound_maxi = 9
    
    def unwhite(Im, mini=False):
        if mini:
            return (Im * (bound_maxi - bound_mini) + bound_mini)*STD[64-32:64+32,64-32:64+32] + MEAN[64-32:64+32,64-32:64+32]
        else:
            return (Im * (bound_maxi - bound_mini) + bound_mini)*STD + MEAN
    def unwhite_cuda(Im, mini=False):
        if mini:
            return (Im * (bound_maxi - bound_mini) + bound_mini)*STD_cuda[64-32:64+32,64-32:64+32] + MEAN_cuda[64-32:64+32,64-32:64+32]
        else:
            return (Im * (bound_maxi - bound_mini) + bound_mini)*STD_cuda + MEAN_cuda
    
    def whitening(Im,mini=False): #mini is True when we need to crop (ie 64x64 images)
        if mini:
            return (Im-MEAN[64-32:64+32,64-32:64+32])/STD[64-32:64+32,64-32:64+32]
        else:
            return (Im-MEAN)/STD
    def whitening_complete(Im, mini=False): #mini is True when we need to crop (ie 64x64 images)
        return (whitening(Im, mini)-bound_mini)/(bound_maxi-bound_mini)
    def whitening_cuda(Im, mini=False): #mini is True when we need to crop (ie 64x64 images)
        if mini:
            return (Im-MEAN_cuda[64-32:64+32,64-32:64+32])/STD_cuda[64-32:64+32,64-32:64+32]
        else:
            return (Im-MEAN_cuda)/STD_cuda
    def whitening_complete_cuda(Im, mini=False): #mini : when we need to crop (ie 64x64 images)
        return (whitening_cuda(Im, mini)-bound_mini)/(bound_maxi-bound_mini)

In [ ]:
if white_gen:
    eq = whitening_complete
    eq_torch = whitening_complete_cuda
    uneq = unwhite
    uneq_torch = unwhite_cuda
else:
    eq = logeq2
    eq_torch = logeq2_torch
    uneq = delogeq2
    uneq_torch = delogeq2

In [ ]:
wgan = WGAN_GP(nb_channels=1, cud=False, n_latent=nz, eq=eq_torch, uneq=uneq_torch, white=white_gen)
wgan.load_model_G(path_models_W + name_model_W)
best_gen = wgan.G.to(device)
best_gen = best_gen.eval()

In [ ]:
def generateur_GAN(params_GAN):
    if type(params_GAN) == list:
        v_latent = torch.reshape(torch.tensor(params_GAN, device=device), (1,nz,1,1))
    elif type(params_GAN) == np.ndarray:
        v_latent = torch.reshape(torch.tensor(params_GAN.astype(np.float32), device=device), (1,nz,1,1))
    elif type(params_GAN) == torch.Tensor:
        v_latent = torch.reshape(torch.tensor(params_GAN, device=device), (1,nz,1,1))
    else:    
        v_latent = torch.reshape(torch.tensor([val for val in params_GAN.values()], device=device), (1,nz,1,1))
    im = best_gen(v_latent)[0].detach()[0,:,:]
    #im = np.array((0.5 + im /2).cpu())
    im = np.array(im.cpu())
    return im
v_latent_init = torch.randn(64,nz,1,1, device=device)
def generateur_GAN_vect(params_GAN):
    v_latent = torch.reshape(torch.tensor(params_GAN, device=device), (1,nz,1,1)).float()
    
    im = best_gen(v_latent)[0].detach()[0,:,:]
    #im = np.array((0.5 + im /2).cpu())
    im = np.array(im.cpu())
    return im

In [ ]:
def complete_generator(z, logvis=False):
    if logvis:
        return viseq(uneq(generateur_GAN(z)))
    else:
        return uneq(generateur_GAN(z))

# Denoising

In [ ]:
lamb=0.002 # Regularization on z
#lamb=0.000

for frame in L_frames:
    print("==============================")
    print("==============================")
    print(frame)
    # Show frame to denoise
    if white_gen:
        img_to_find = eq(PSF_og[frame], mini=True)
    else:
        img_to_find = eq(PSF_og[frame])
    
    plt.imshow(img_to_find)
    plt.title(frame)
    plt.show()
    
    # Define function to optimize
    def func_to_opti1(par):
        im_test = crop(generateur_GAN(par))
        return np.sum(np.nan_to_num(img_to_find - im_test)**2) + lamb*np.sum(par**2)

    img_to_find_tensor = torch.Tensor(img_to_find).to(device)

    def deriv_func(par): #Derivatives
        par_tensor = torch.Tensor(par).to(device)
        par_tensor.requires_grad=True
        res = torch.sum(torch.nan_to_num((crop_tensor4(best_gen(torch.reshape(par_tensor, (1,nz,1,1)))) - img_to_find_tensor))**2)
        return np.array(grad(outputs=res, inputs=par_tensor)[0].detach().cpu()) + 2*lamb*par
    
    best_gen.to(device)
    
    best_MSE = 1000000000000000000
    best_z = None

    for i in range(10): # We try from 10 different starting points

        z = np.random.randn(nz) #Starting point

        L_suivi = [z] # If we want to follow the evolution

        def func_callback(xk):
            L_suivi.append(xk)

        # Minimization with BFGS
        res = scipy.optimize.minimize(func_to_opti1, z, method="BFGS", jac=deriv_func, callback=func_callback)
        
        # Compute MSE without the regularization
        if white_gen:
            new_MSE = MSE(logeq2(uneq(img_to_find,mini=True)), crop(logeq2(uneq(generateur_GAN_vect(res.x)))))
        else:
            new_MSE = MSE(img_to_find, crop(generateur_GAN_vect(res.x)))
            
        # Keep the best (for the MSE criterium) z
        if new_MSE < best_MSE:
            best_MSE = new_MSE
            best_z = res.x

    # Show results
    z1 = best_z
    plt.imshow(generateur_GAN_vect(z1))
    plt.title("Best")
    plt.show()
    print("\n",new_MSE, z1)
    
    # Contrast between 0 and 1
    plt.imshow(img_to_find, vmin=0, vmax=1)
    plt.title("True")
    plt.show()

    plt.imshow(crop(generateur_GAN_vect(z1)), vmin=0, vmax=1)
    plt.title("Best")
    plt.show()

    # Maximal contrast
    mini = min(np.min(img_to_find), np.min(crop(generateur_GAN_vect(z1))))
    maxi = max(np.max(img_to_find), np.max(crop(generateur_GAN_vect(z1))))
    plt.imshow(img_to_find, vmin=mini, vmax=maxi) 
    plt.title("True")
    plt.show()

    plt.imshow(crop(generateur_GAN_vect(z1)), vmin=mini, vmax=maxi)
    plt.title("Best")
    plt.show()

    # No cropping
    plt.imshow(generateur_GAN_vect(z1), vmin=0, vmax=1)
    plt.title("Best")
    plt.show()
    plt.imshow(generateur_GAN_vect(z1))
    plt.title("Best")
    plt.show()

    # Show errors
    vminimaxi = np.max(abs(img_to_find - crop(generateur_GAN_vect(z1))))
    plt.imshow((img_to_find - crop(generateur_GAN_vect(z1))), cmap="bwr", vmin=-vminimaxi, vmax=vminimaxi)
    plt.title("Errors")
    plt.colorbar()
    plt.show()
    if white_gen:
        plt.imshow(viseq(uneq(img_to_find, mini=True)), vmin=0, vmax=1)
    else:
        plt.imshow(viseq(uneq(img_to_find)), vmin=0, vmax=1)
    plt.title("True")
    plt.show()

    # Show with better log-scaling (risk of nan)
    if white_gen:
        plt.imshow(viseq(uneq(crop(generateur_GAN_vect(z1)), mini=True)), vmin=0, vmax=1)
    else:
        plt.imshow(viseq(uneq(crop(generateur_GAN_vect(z1)))), vmin=0, vmax=1)
    plt.title("Best")
    plt.show()


    # Show with better log-scaling (risk of nan)
    if white_gen:
        plt.imshow(logeq2(uneq(img_to_find, mini=True)), vmin=0, vmax=1)
    else:
        plt.imshow(logeq2(uneq(img_to_find)), vmin=0, vmax=1)
    plt.title("True")
    plt.show()

    
    # Show with log-equalization
    if white_gen:
        plt.imshow(logeq2(uneq(crop(generateur_GAN_vect(z1)), mini=True)), vmin=0, vmax=1)
    else:
        plt.imshow(logeq2(uneq(crop(generateur_GAN_vect(z1)))), vmin=0, vmax=1)
    plt.title("Best")
    plt.show()
    
    # Show z
    plt.bar(list(range(nz)),z1)
    plt.show()
    
    # Storing
    PSF_gen[frame] = uneq(generateur_GAN_vect(z1))
    z_gen[frame] = z1
    best_MSEs[frame] = best_MSE

## Middle image : if there are two frames, we interpolate between both

In [ ]:
if nb_frames > 1:
    
    for frame in ["K1", "K2"]:
        
        print("==============================")
        print("==============================")
        print(frame)
        
        # Get the z that were found previously during the denoising
        z1 = z_gen[frame+"_f1"]
        z2 = z_gen[frame+"_f2"]
        
        # Show both z
        plt.bar(list(range(nz)),z1, width=0.4)
        plt.bar(np.array(list(range(nz)))+0.4,z2, width=0.4)
        plt.title(frame)
        plt.show()
        zmid = (z1+z2)/2
        
        # Show middle psf
        plt.imshow(generateur_GAN_vect(zmid), vmin=0, vmax=1)
        plt.title("Mid")
        plt.show()
                
        # Show cropped middle psf
        plt.imshow(crop(generateur_GAN_vect(zmid)), vmin=0, vmax=1)
        plt.title("Mid - cropped")
        plt.show()

        # Show it on the best log-scale (risk of nan)
        plt.imshow(viseq(delogeq2(crop(generateur_GAN_vect(zmid)))), vmin=0, vmax=1)
        plt.title("Mid")
        plt.show()
        
        
        
        maximini = max(np.max(abs(crop(generateur_GAN_vect(z1)) - crop(generateur_GAN_vect(zmid)))), np.max(abs(crop(generateur_GAN_vect(z2)) - crop(generateur_GAN_vect(zmid)))))
        # Show difference between middle image and the first one
        plt.imshow(crop(generateur_GAN_vect(z1)) - crop(generateur_GAN_vect(zmid)), cmap="bwr", vmin= -maximini, vmax= maximini)
        plt.title("Diff z1 - zmid")
        plt.colorbar()
        plt.show()

        # Show difference between middle image and the second one
        plt.imshow(-crop(generateur_GAN_vect(z2)) + crop(generateur_GAN_vect(zmid)), cmap="bwr", vmin= -maximini, vmax= maximini)
        plt.title("Diff zmid - z2")
        plt.colorbar()
        plt.show()

        # Show difference between the two initial images
        maximini = np.max(abs(crop(generateur_GAN_vect(z1)) - crop(generateur_GAN_vect(z2))))
        plt.imshow(crop(generateur_GAN_vect(z2)) - crop(generateur_GAN_vect(z1)), cmap="bwr", vmin= -maximini, vmax= maximini)
        plt.title("Diff z2 - z1")
        plt.colorbar()
        plt.show()

        
        # Storing
        PSF_gen[frame] = uneq(generateur_GAN_vect(zmid))
        z_gen[frame] = zmid

# Cancelling the equalization

In [ ]:
PSF_genADU = {} # the PSF in ADU

In [ ]:
def affine_solve(data,abscisse): # Function used to rescale the PSF values
    N = data.shape[0]
    
    assert abscisse.shape[0] == N
        
    st=0
    st2=0
    sy=0
    sy2=0
    
    st = np.sum(abscisse)
    st2 = np.sum(abscisse**2)
    sy = np.sum(data)
    sty = np.sum(data*abscisse)
        
    i_delta = 1 / (N*st2 - st**2)
    intercept = i_delta * (st2 * sy - st * sty)
    slope =  i_delta * (N * sty - st * sy)
    return intercept, slope



In [ ]:
for frame in ["K1", "K2"]:
    if nb_frames > 1:
        max_ref = (maxs[frame+"_f1"] + maxs[frame+"_f2"])/2 # Get back the maximal value before normalization
        alpha_ref = (alphas[frame+"_f1"] + alphas[frame+"_f2"])/2 # Get back the sum value before normalization
    else:
        max_ref = maxs[frame]# Get back the maximal value before normalization
        alpha_ref = alphas[frame]# Get back the sum value before normalization
    
    
    if nb_frames > 1:
        the_old_PSF = (PSF_old[frame+"_f1"] + PSF_old[frame+"_f2"])/2
        plt.imshow(viseq(the_old_PSF))
        plt.title("Mean of two initial psf")
        plt.show()
    else:
        the_old_PSF = PSF_old[frame]
        plt.imshow(viseq(the_old_PSF))
        plt.title("Initial psf")
        plt.show()
    
    b, a = affine_solve(the_old_PSF.flatten(), crop(PSF_gen[frame]).flatten()) # Get the scaling factors
    print(frame)
    print(a, alpha_ref, max_ref/np.max(PSF_gen[frame])) # Compare the scaling factors to the stored values
    if nb_frames == 1:
        print(b, shifts[frame], "\n") # It should be ~ the same
    PSF_genADU[frame] = a*PSF_gen[frame] # Rescaling
    
    # Show histogram of pixel values to check if it is centered on 0
    plt.hist(PSF_gen[frame].flatten()[PSF_gen[frame].flatten() < 0.0001], bins=100) # Values for the generated psf
    plt.title("Generated")
    plt.show() 
    plt.hist(PSF_genADU[frame].flatten()[PSF_genADU[frame].flatten() < 200], bins=100) # Same thing in ADU
    plt.title("Generated - ADU")
    plt.show()
    
    plt.hist(the_old_PSF.flatten()[the_old_PSF.flatten() < 200], bins=100) # Comparison with the noisy psf
    plt.title("Noisy")
    plt.show()

# Last step : saving the result

In [ ]:
hdr = fits.Header()
hdr['K12_fra'] = 'First : K1; Second : K2'
for frame in L_frames:
    print(frame, best_MSEs[frame])
    hdr["MSE_" + frame] = best_MSEs[frame]

if "2025" in name_model_W:
    year = "2025"
else:
    year = "2024"

L_W = mode_eq[0].upper()

data128 = np.array([PSF_genADU["K1"], PSF_genADU["K2"]])
hdu = fits.PrimaryHDU(data=data128, header=hdr)
hdul = fits.HDUList([hdu])
hdul.writeto(to_test+'_gen'+L_W+year+'_128.fits')

data64 = np.array([crop(PSF_genADU["K1"]), crop(PSF_genADU["K2"])])
hdu = fits.PrimaryHDU(data=data64, header=hdr)
hdul = fits.HDUList([hdu])
hdul.writeto(to_test+'_gen'+L_W+year+'_64.fits')